# FSL, MRtrix3, and DIPY: A Comparison

Three major toolboxes dominate dMRI analysis. This notebook shows you what each one is, installs/checks them, and runs a simple command in each so you can see the interface side-by-side before we go deeper.

---

## The Big Picture

| | FSL | MRtrix3 | DIPY |
|---|---|---|---|
| **Type** | CLI suite | CLI suite + mrview | Python library |
| **Language** | C++ (Python wrappers) | C++ (Python wrappers) | Pure Python |
| **License** | Free, non-commercial | Open source (MPL 2.0) | BSD |
| **Strengths** | Eddy correction, GLM, bedpostX | CSD, iFOD2 tractography, SIFT | Transparency, flexibility, teaching |
| **Weaknesses** | Closed-source eddy; GUI limited | Steeper learning curve | Slower for large datasets |
| **Key papers** | Smith et al. 2004 | Tournier et al. 2019 | Garyfallidis et al. 2014 |
| **Best for** | Clinical/group studies | Research tractography | Custom pipelines, learning |

**The key insight:** these tools are *complementary*, not competing. Most real pipelines combine all three.

In [ ]:
# ── Check that all three toolboxes are available ─────────────────────────────
import sys
sys.path.insert(0, '../../scripts')
from utils import check_all_tools

check_all_tools()
# If something shows ✗, see the environment setup notebook.

---

## 1. Reading a NIfTI header: three ways

The same file, the same information, three interfaces. Let's start as simple as possible.

In [ ]:
# ─── Common setup ────────────────────────────────────────────────────────────
from pathlib import Path
import subprocess

data_dir = Path('../../data/hcp/100307/T1w/Diffusion')
dwi_file = str(data_dir / 'data.nii.gz')
print(f'Using: {dwi_file}\n')

In [ ]:
# ─── [FSL] fslinfo ────────────────────────────────────────────────────────────
print('=== FSL: fslinfo ===')
result = subprocess.run(['fslinfo', dwi_file],
                        capture_output=True, text=True)
print(result.stdout)

In [ ]:
# ─── [MRtrix3] mrinfo ────────────────────────────────────────────────────────
print('=== MRtrix3: mrinfo ===')
result = subprocess.run(['mrinfo', dwi_file],
                        capture_output=True, text=True)
print(result.stdout)

In [ ]:
# ─── [DIPY] nibabel ───────────────────────────────────────────────────────────
import nibabel as nib
import numpy as np

print('=== DIPY / nibabel ===')
img = nib.load(dwi_file)
print(f'  Shape      : {img.shape}')
print(f'  Voxel size : {img.header.get_zooms()}')
print(f'  Data type  : {img.get_data_dtype()}')
print(f'  Affine     :\n{img.affine}')

> **Notice:** FSL and MRtrix3 print from the command line; DIPY gives you Python objects you can manipulate directly. Neither approach is better — they reflect different philosophies.

---

## 2. Converting between formats

MRtrix3 uses its own `.mif` format which embeds gradient info **inside** the file. Converting once at the start of a pipeline avoids many common errors.

In [ ]:
# ─── [MRtrix3] mrconvert: NIfTI + bvals/bvecs → .mif ────────────────────────
mif_out = str(data_dir / 'data.mif')

result = subprocess.run([
    'mrconvert',
    dwi_file,
    mif_out,
    '-fslgrad',
    str(data_dir / 'bvecs'),
    str(data_dir / 'bvals'),
    '-force'          # overwrite if exists
], capture_output=True, text=True)

print(result.stdout or result.stderr)

# Confirm the gradient table is embedded
result2 = subprocess.run(['mrinfo', mif_out, '-grad'],
                         capture_output=True, text=True)
print('\nEmbedded gradient table (first 5 rows):')
lines = result2.stdout.strip().split('\n')
for line in lines[:5]:
    print(' ', line)

---

## 3. Viewing data

Each toolkit has its own viewer. You cannot call GUI tools from a notebook, but you can launch them:

In [ ]:
# ─── Launch FSLeyes (FSL's viewer) ───────────────────────────────────────────
# subprocess.Popen(['fsleyes', dwi_file])   # uncomment to open
print('[FSL]     $ fsleyes', dwi_file)

# ─── Launch mrview (MRtrix3's viewer) ────────────────────────────────────────
# subprocess.Popen(['mrview', mif_out])     # uncomment to open
print('[MRtrix3] $ mrview', mif_out)

# ─── In-notebook viewing with matplotlib (DIPY approach) ─────────────────────
import matplotlib.pyplot as plt

data = img.get_fdata()
z = data.shape[2] // 2

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
vol_indices = [0, 30, 90]   # b=0, b≈1000, b≈2000
for ax, vi in zip(axes, vol_indices):
    ax.imshow(data[:, :, z, vi].T, cmap='gray', origin='lower')
    ax.set_title(f'Volume {vi} | b≈{int(np.loadtxt(data_dir / "bvals")[vi])} s/mm²')
    ax.axis('off')
fig.suptitle('[DIPY] In-notebook slice view — no GUI required', fontsize=11)
plt.tight_layout()
plt.show()

---

## 4. Gradient table loading: three interfaces

The gradient table is central to every step. Here is how each tool reads it.

In [ ]:
# ─── [DIPY] GradientTable object ─────────────────────────────────────────────
from dipy.io.gradients import read_bvals_bvecs
from dipy.core.gradients import gradient_table

bvals, bvecs = read_bvals_bvecs(
    str(data_dir / 'bvals'),
    str(data_dir / 'bvecs')
)
gtab = gradient_table(bvals, bvecs)

print('[DIPY] GradientTable summary')
print(f'  b0s_mask  : {gtab.b0s_mask.sum()} volumes with b≈0')
print(f'  bvals     : {np.unique(np.round(gtab.bvals, -2))}')
print(f'  bvecs shape: {gtab.bvecs.shape}')

# ─── [FSL] bvals / bvecs are plain text files (read directly) ─────────────────
print('\n[FSL] bvals (first 10):', np.loadtxt(data_dir / 'bvals')[:10].astype(int))

# ─── [MRtrix3] mrinfo extracts gradient table from .mif ───────────────────────
print('\n[MRtrix3] mrinfo -grad output (first 3 rows):')
r = subprocess.run(['mrinfo', mif_out, '-grad'],
                   capture_output=True, text=True)
for line in r.stdout.strip().split('\n')[:3]:
    print(' ', line)

---

## Summary: Which tool when?

| Task | Recommended tool | Why |
|---|---|---|
| Eddy current + motion correction | **FSL eddy** | Most validated, handles multi-shell |
| Susceptibility distortion | **FSL topup** | Gold standard |
| Fibre response estimation | **MRtrix3 dwi2response** | dhollander algorithm best available |
| CSD / FOD estimation | **MRtrix3 dwi2fod** | Native support for msmt-CSD |
| Whole-brain tractography | **MRtrix3 tckgen** | iFOD2 + ACT is state-of-the-art |
| Streamline filtering | **MRtrix3 tcksift2** | Corrects tractography biases |
| DTI metrics (quick) | **FSL dtifit** or **DIPY** | Both fast; DIPY gives Python objects |
| Custom analysis / prototyping | **DIPY** | Full Python API, easy to modify |
| Visualisation | **MRtrix3 mrview** or **FSLeyes** | Both excellent; pick your preference |

**Next**: [Setting up your HCP data →](02_data_setup.ipynb)